# PANDA WSI MIL – workflow

- **Labels:** `data/raw/train.csv` (image_id, isup_grade)
- **Splits:** `data/splits/panda_5fold_stratified.csv`
- **Features:** `data/trident_out/.../features_uni_v2/*.h5`

**Setup (run once):** Spusti nasledujúcu bunku, aby sa balíčky nainštalovali do prostredia tohto Jupyter image. Ak `openslide-python` zlyhá (chýba systémová knižnica), nainštaluj aspoň ostatné a v importoch zakomentuj `import openslide`.

In [ ]:
# Run this cell once to install packages into the current Jupyter image environment
# (e.g. when using MetaCentrum Interactive Apps → Jupyter)
%pip install --quiet openslide-python scikit-image pandas numpy matplotlib Pillow h5py scikit-learn plotly torch

In [ ]:
import os
from pathlib import Path

# Project root: env MIL, or walk up from cwd until we see data/
cwd = Path.cwd()
if os.environ.get("MIL"):
    MIL_ROOT = Path(os.environ["MIL"]).resolve()
else:
    MIL_ROOT = cwd
    for _ in range(5):
        if (MIL_ROOT / "data").exists():
            break
        MIL_ROOT = MIL_ROOT.parent
print("MIL_ROOT:", MIL_ROOT)

In [ ]:
# PANDA: load WSIs (openslide or skimage) + viz
import openslide
try:
    import skimage.io
except ModuleNotFoundError:
    skimage = None  # pip install scikit-image

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import PIL
from IPython.display import Image, display

In [ ]:
# Paths and labels (Kaggle-style names; data lives under MIL_ROOT)
# Try cluster paths (/storage or /auto mount on MetaCentrum)
for root in [Path("/storage/brno2/home/filipsec/MIL"), Path("/auto/brno2/home/filipsec/MIL")]:
    wsi_dir = root / "data" / "raw" / "train_images"
    if wsi_dir.exists():
        MIL_ROOT = root
        break

data_dir = MIL_ROOT / "data" / "raw" / "train_images"
mask_dir = MIL_ROOT / "data" / "raw" / "train_label_masks"
train_labels = pd.read_csv(MIL_ROOT / "data" / "raw" / "train.csv").set_index("image_id")
print("MIL_ROOT:", MIL_ROOT)
print("data_dir:", data_dir)

In [ ]:
# train_labels already loaded above; train = same data without index for compatibility
train = train_labels.reset_index()
print(train.shape)
print(train["isup_grade"].value_counts().sort_index())

In [ ]:
# OpenSlide: always close the slide when done (use try/finally or context manager)
data_dir = MIL_ROOT / "data" / "raw" / "train_images"
labels = pd.read_csv(MIL_ROOT / "data" / "raw" / "train.csv")
path = data_dir / f"{labels['image_id'].iloc[0]}.tiff"
if path.exists():
    biopsy = openslide.OpenSlide(str(path))
    try:
        # do something with the slide here
        print("Levels:", biopsy.level_count, "Dimensions:", biopsy.dimensions)
    finally:
        biopsy.close()
else:
    print("Not found:", path)

In [ ]:
def display_images(image_ids, data_dir=None, max_size=512, ncols=4):
    """Display thumbnails of raw WSI .tiff slides (before feature extraction)."""
    if data_dir is None:
        data_dir = MIL_ROOT / "data" / "raw" / "train_images"
    data_dir = Path(data_dir)
    n = len(image_ids)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    if n == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes.reshape(1, -1)
    for ax in axes.flat:
        ax.axis("off")
    for i, img_id in enumerate(image_ids):
        ax = axes.flat[i]
        path = data_dir / f"{img_id}.tiff"
        if not path.exists():
            ax.set_title(f"{img_id[:8]}... (not found)")
            continue
        try:
            slide = openslide.OpenSlide(str(path))
            try:
                thumb = slide.get_thumbnail((max_size, max_size))
                ax.imshow(thumb)
            finally:
                slide.close()
        except Exception as e:
            ax.set_title(f"{img_id[:8]}... (error)")
            ax.text(0.5, 0.5, str(e)[:50], ha="center", va="center", fontsize=8)
            continue
        label = train_labels.loc[img_id, "isup_grade"] if img_id in train_labels.index else "?"
        ax.set_title(f"{img_id[:8]}... ISUP={label}")
    plt.tight_layout()
    plt.show()

In [ ]:
# 21 slides that did not produce features (missing_21_slides.csv) – visualize raw .tiff before preprocessing
MISSING_21 = [
    '163fabe883fcd17de4f899ca8ede45a8', '1da6dabc916a5258e4bf7b9eb425cbda', '3790f55cad63053e956fb73027179707',
    '46bf558b24bcdfe1f3ae711b02b53a63', '5aa7cc54ea328d8b61385a4cd2ff2352', '615b8a4e40699ea384e5f518ac0c68b4',
    '67e855cca15a9930d065bf9f14241b41', '6844032932a3939565e936a523174b6d', '70d813b5e09d4fdac98c761c15c28a62',
    '7ec4b5a5d165debf9e06aba787d1ab7a', '8423df90dddd142668c18b6ab5ff7f17', '88e65e1d716893e6efdbd2464b3686fa',
    '8b012a7302f32e3f7d03c59473992462', '8b301972309a5509bbb5946b9f133e37', 'af994df40907edf1c3aefb45df954add',
    'c14efea7c94be982c8045ff9d3c16266', 'c93b8d5f45fb7dea353560cba2eebdde', 'da09fe1419dcf0c2e16c0b2a43979c43',
    'e04e11360525f7d13aaa9d4bba16ea7e', 'e0c2ac1e8919079234c7d0b9df13adff', 'eb879c95e942688b2fde3061e313e5b8',
]

# Diagnose: ensure display_images gets the correct data_dir (pass explicitly)
print("data_dir:", data_dir)
print("exists:", data_dir.exists())
if data_dir.exists():
    sample = list(data_dir.glob("*.tiff"))[:3]
    print("sample .tiff:", [p.name for p in sample])
    found = sum(1 for sid in MISSING_21 if (data_dir / f"{sid}.tiff").exists())
    print(f"MISSING_21: {found}/21 .tiff files found")
display_images(MISSING_21, data_dir=data_dir)

In [ ]:
def print_slide_details(biopsy):
    """Print level count and dimensions of an OpenSlide object."""
    print("Levels:", biopsy.level_count, "Dimensions:", biopsy.dimensions)

# Missing 21 slides (from data/lists/missing_21_slides.csv)
example_slides = [
    '163fabe883fcd17de4f899ca8ede45a8',
    '1da6dabc916a5258e4bf7b9eb425cbda',
    '3790f55cad63053e956fb73027179707',
    '46bf558b24bcdfe1f3ae711b02b53a63',
    '5aa7cc54ea328d8b61385a4cd2ff2352',
    '615b8a4e40699ea384e5f518ac0c68b4',
    '67e855cca15a9930d065bf9f14241b41',
    '6844032932a3939565e936a523174b6d',
    '70d813b5e09d4fdac98c761c15c28a62',
    '7ec4b5a5d165debf9e06aba787d1ab7a',
    '8423df90dddd142668c18b6ab5ff7f17',
    '88e65e1d716893e6efdbd2464b3686fa',
    '8b012a7302f32e3f7d03c59473992462',
    '8b301972309a5509bbb5946b9f133e37',
    'af994df40907edf1c3aefb45df954add',
    'c14efea7c94be982c8045ff9d3c16266',
    'c93b8d5f45fb7dea353560cba2eebdde',
    'da09fe1419dcf0c2e16c0b2a43979c43',
    'e04e11360525f7d13aaa9d4bba16ea7e',
    'e0c2ac1e8919079234c7d0b9df13adff',
    'eb879c95e942688b2fde3061e313e5b8',
]

for case_id in example_slides:
    path = data_dir / f"{case_id}.tiff"
    if not path.exists():
        print(f"Skip (file not found): {case_id}\n")
        continue
    biopsy = openslide.OpenSlide(str(path))
    try:
        print_slide_details(biopsy)
    finally:
        biopsy.close()
    if case_id in train_labels.index:
        print(f"ISUP grade: {train_labels.loc[case_id, 'isup_grade']}")
        print(f"Gleason score: {train_labels.loc[case_id, 'gleason_score']}")
    else:
        print("(no label in train.csv)")
    print("\n")

In [ ]:
# Splits (if generated)
splits_path = MIL_ROOT / "data" / "splits" / "panda_5fold_stratified.csv"
if splits_path.exists():
    splits = pd.read_csv(splits_path)
    print(splits["fold"].value_counts().sort_index())
else:
    print("Run: python scripts/make_panda_5fold_splits.py")

In [ ]:
import h5py

feat_dir = MIL_ROOT / "data" / "trident_out" / "panda_uni_v2_grandqc_20x_256_ov0" / "20x_256px_0px_overlap" / "features_uni_v2"
if not feat_dir.exists():
    feat_dir = MIL_ROOT / "data" / "trident_out_makeup_otsu" / "20x_256px_0px_overlap" / "features_uni_v2"
h5_files = list(feat_dir.glob("*.h5"))[:3] if feat_dir.exists() else []
for p in h5_files:
    with h5py.File(p, "r") as f:
        print(p.name, list(f.keys()), f["features"].shape if "features" in f else "")

## MIL: load one bag and label

Next: build Dataset that loads .h5 + merge with train.csv by image_id; then train/val loop.

In [ ]:
# Load .h5 features for only these 21 slides (missing in Virchow2, present in uni_v2)
from tqdm import tqdm

def load_h5_features(feat_dir, max_slides=200, key='features', slide_ids=None):
    """Load patch features. If slide_ids given, load only those."""
    feat_path = Path(feat_dir)
    if slide_ids is not None:
        h5s = [feat_path / f"{sid}.h5" for sid in slide_ids]
        h5s = [h for h in h5s if h.exists()]
    else:
        h5s = sorted(feat_path.glob('*.h5'))[:max_slides]
    all_feats, all_sids = [], []
    for h5 in tqdm(h5s, desc='Loading'):
        with h5py.File(h5, 'r') as f:
            ft = f[key][:]
        all_feats.append(ft)
        all_sids.extend([h5.stem] * ft.shape[0])
    if not all_feats:
        return np.array([]).astype(np.float32).reshape(0, 0), np.array([])
    feats = np.concatenate(all_feats, axis=0).astype(np.float32)
    sids = np.array(all_sids)
    print(f'{feats.shape[0]} patches from {len(h5s)} slides, feat_dim={feats.shape[1]}')
    return feats, sids

# Diagnose: does feat_dir exist? Are any of the 21 .h5 files there?
MISSING_21 = [
    '163fabe883fcd17de4f899ca8ede45a8',
    '1da6dabc916a5258e4bf7b9eb425cbda',
    '3790f55cad63053e956fb73027179707',
    '46bf558b24bcdfe1f3ae711b02b53a63',
    '5aa7cc54ea328d8b61385a4cd2ff2352',
    '615b8a4e40699ea384e5f518ac0c68b4',
    '67e855cca15a9930d065bf9f14241b41',
    '6844032932a3939565e936a523174b6d',
    '70d813b5e09d4fdac98c761c15c28a62',
    '7ec4b5a5d165debf9e06aba787d1ab7a',
    '8423df90dddd142668c18b6ab5ff7f17',
    '88e65e1d716893e6efdbd2464b3686fa',
    '8b012a7302f32e3f7d03c59473992462',
    '8b301972309a5509bbb5946b9f133e37',
    'af994df40907edf1c3aefb45df954add',
    'c14efea7c94be982c8045ff9d3c16266',
    'c93b8d5f45fb7dea353560cba2eebdde',
    'da09fe1419dcf0c2e16c0b2a43979c43',
    'e04e11360525f7d13aaa9d4bba16ea7e',
    'e0c2ac1e8919079234c7d0b9df13adff',
    'eb879c95e942688b2fde3061e313e5b8',
]

# Diagnose: 0it usually means feat_dir missing or no .h5 for these IDs
print("feat_dir:", feat_dir)
print("exists:", feat_dir.exists())
if feat_dir.exists():
    all_h5 = list(feat_dir.glob("*.h5"))
    print("total .h5 files:", len(all_h5))
    found = [sid for sid in MISSING_21 if (feat_dir / f"{sid}.h5").exists()]
    print("of 21 MISSING_21, found:", len(found), found[:3] if found else "[]")
else:
    print("feat_dir does not exist – run on cluster or ensure trident features are extracted")

feats, sids = load_h5_features(feat_dir, slide_ids=MISSING_21)

In [ ]:
# Example: one slide as bag
if h5_files and len(train_labels):
    slide_id = h5_files[0].stem
    with h5py.File(h5_files[0], "r") as f:
        feats = f["features"][:]
    label = int(train_labels.loc[slide_id, "isup_grade"]) if slide_id in train_labels.index else None
    print(f"Slide {slide_id}: features {feats.shape}, isup_grade={label}")

## PyTorch MIL: Dataset + model + training

- **Dataset:** one bag per slide = one `.h5` file → `features` array `[N_patches, feat_dim]`, label = `isup_grade` (0–5).
- **Model:** patch features → attention pooling → slide-level logits (6 classes).
- **Training:** cross-entropy, optional 5-fold from `splits`, validation accuracy / balanced accuracy / Cohen's kappa.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import h5py

class H5BagDataset(Dataset):
    """One slide = one bag: load features from .h5 and label from train_labels (isup_grade 0-5)."""
    def __init__(self, feat_dir, slide_ids, train_labels, key="features"):
        self.feat_dir = Path(feat_dir)
        self.slide_ids = list(slide_ids)
        self.labels = train_labels
        self.key = key

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        slide_id = self.slide_ids[idx]
        h5_path = self.feat_dir / f"{slide_id}.h5"
        with h5py.File(h5_path, "r") as f:
            feats = torch.from_numpy(f[self.key][:]).float()
        label = int(self.labels.loc[slide_id, "isup_grade"])
        return feats, label, slide_id

In [ ]:
def collate_bags(batch):
    """Collate to padded batch: (feats [B,N_max,D], mask [B,N_max], labels [B], ids)."""
    feats_list = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    ids = [b[2] for b in batch]
    N_max = max(f.size(0) for f in feats_list)
    D = feats_list[0].size(1)
    device = feats_list[0].device
    feats = torch.zeros(len(batch), N_max, D, dtype=feats_list[0].dtype)
    mask = torch.zeros(len(batch), N_max, dtype=torch.bool)
    for i, f in enumerate(feats_list):
        n = f.size(0)
        feats[i, :n] = f
        mask[i, :n] = True
    return feats, mask, labels, ids

In [ ]:
class AttentionMIL(nn.Module):
    """Attention-based MIL: patch features -> attention (masked) -> weighted sum -> classifier."""
    def __init__(self, feat_dim, num_classes=6, hidden=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, x, mask=None):
        # x: [B, N, D], mask: [B, N] True where valid
        h = self.encoder(x)
        a = self.attention(h).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, -1e9)
        a = torch.softmax(a, dim=1)
        bag = (a.unsqueeze(-1) * h).sum(dim=1)
        return self.classifier(bag)

In [ ]:
# Build train/val slide lists: use splits if available, else 80/20 by isup_grade
feat_dir = MIL_ROOT / "data" / "trident_out" / "panda_uni_v2_grandqc_20x_256_ov0" / "20x_256px_0px_overlap" / "features_uni_v2"
if not feat_dir.exists():
    feat_dir = MIL_ROOT / "data" / "trident_out_makeup_otsu" / "20x_256px_0px_overlap" / "features_uni_v2"

available_h5 = {p.stem for p in feat_dir.glob("*.h5")} if feat_dir.exists() else set()
# Only slides that have both labels and features
slide_ids = [sid for sid in train_labels.index if sid in available_h5]
labels_subset = train_labels.loc[slide_ids]

if splits_path.exists() and "fold" in splits.columns:
    fold = 0
    train_ids = splits[splits["fold"] != fold]["image_id"].tolist()
    val_ids = splits[splits["fold"] == fold]["image_id"].tolist()
    train_ids = [i for i in train_ids if i in available_h5]
    val_ids = [i for i in val_ids if i in available_h5]
else:
    from sklearn.model_selection import train_test_split
    train_ids, val_ids = train_test_split(
        slide_ids, test_size=0.2, random_state=42, stratify=labels_subset["isup_grade"]
    )

print("Train slides:", len(train_ids), "Val slides:", len(val_ids))

In [ ]:
# DataLoaders and model
train_ds = H5BagDataset(feat_dir, train_ids, train_labels)
val_ds = H5BagDataset(feat_dir, val_ids, train_labels)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, collate_fn=collate_bags)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_bags)

# Infer feat_dim from first file
with h5py.File(feat_dir / f"{train_ids[0]}.h5", "r") as f:
    feat_dim = f["features"].shape[1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionMIL(feat_dim=feat_dim, num_classes=6, hidden=128).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Training loop (few epochs as example)
# If you get NameError, run the "DataLoaders and model" cell above first.
try:
    _ = model
except NameError:
    train_ds = H5BagDataset(feat_dir, train_ids, train_labels)
    val_ds = H5BagDataset(feat_dir, val_ids, train_labels)
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, collate_fn=collate_bags)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_bags)
    with h5py.File(feat_dir / f"{train_ids[0]}.h5", "r") as f:
        _fd = f["features"].shape[1]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AttentionMIL(feat_dim=_fd, num_classes=6, hidden=128).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    print("Built model and loaders.")

model.train()
for epoch in range(3):
    total_loss = 0.0
    for feats, mask, labels, _ in train_loader:
        feats, mask, labels = feats.to(device), mask.to(device), labels.to(device)
        opt.zero_grad()
        logits = model(feats, mask)
        loss = criterion(logits, labels)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} train loss: {total_loss/len(train_loader):.4f}")

# Validation: accuracy and balanced accuracy
model.eval()
all_pred, all_label = [], []
with torch.no_grad():
    for feats, mask, labels, _ in val_loader:
        feats, mask, labels = feats.to(device), mask.to(device), labels.to(device)
        logits = model(feats, mask)
        pred = logits.argmax(dim=1)
        all_pred.append(pred.cpu())
        all_label.append(labels.cpu())
all_pred = torch.cat(all_pred)
all_label = torch.cat(all_label)
acc = (all_pred == all_label).float().mean().item()
print(f"Val accuracy: {acc:.4f}")
from sklearn.metrics import balanced_accuracy_score
print("Val balanced accuracy:", balanced_accuracy_score(all_label.numpy(), all_pred.numpy()))

### Optional: MIL-Lab (ABMIL, TransMIL, CLAM, …)

Ak máš nainštalovaný [MIL-Lab](https://github.com/mahmoodlab/MIL-Lab), môžeš namiesto `AttentionMIL` použiť ich modely. Dáta (feat_dir, train_ids, val_ids, loadery) ostávajú rovnaké. Nižšie: vytvorenie modelu podľa `feat_dim` a tréning s `results_dict["logits"]` / `results_dict["loss"]`.

In [ ]:
# Optional: MIL-Lab – potrebuješ pip install -e . v MIL-Lab repo
try:
    from src.builder import create_model as mil_create_model
except ImportError:
    mil_create_model = None

if mil_create_model is not None and "feat_dim" in dir():
    # Model podľa feat_dim: virchow2=2560, uni_v2=1536
    if feat_dim == 2560:
        model_mil = mil_create_model("abmil.base.virchow2.none", num_classes=6)
    elif feat_dim == 1536:
        model_mil = mil_create_model("abmil.base.uni_v2.none", num_classes=6)
    else:
        model_mil = mil_create_model("abmil.base.uni.none", in_dim=feat_dim, num_classes=6)
    model_mil = model_mil.to(device)
    opt_mil = torch.optim.Adam(model_mil.parameters(), lr=1e-4)

    # Jeden epoch s MIL-Lab forward (feats bez masky – padding nuly)
    model_mil.train()
    for feats, mask, labels, _ in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        opt_mil.zero_grad()
        results_dict, _ = model_mil(feats, loss_fn=criterion, label=labels, return_attention=True)
        results_dict["loss"].backward()
        opt_mil.step()
    print("MIL-Lab one-epoch train done. Use results_dict['logits'] for val.")
else:
    print("MIL-Lab not installed or feat_dim not set. Skip.")